<a href="https://colab.research.google.com/github/vinayprabhu/APEX_reproduce/blob/main/APEX_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Goal

This notebook reproduces the errors encountered when I tried to replicate the paper: "_Computational exploration of global venoms for antimicrobial discovery with Venomics artificial intelligence_". (Paper [link](https://www.nature.com/articles/s41467-025-60051-6) )




In [ ]:
#@title  Let us begin with the basic imports
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from scipy.linalg import block_diag
# Don't do linear algebra in Python without these two lines
np.set_printoptions(suppress=True)
from collections import Counter
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

%precision 3
#############################################
import sys
import importlib
importlib.reload(sys)
#######################
from google.colab import drive
drive.flush_and_unmount()
import os
drive.mount('/gdrive', force_remount=True)
# Enter your own proj_dir here
proj_dir='/gdrive/My Drive/venomics.ai/AI/VENOMICS_UPENN/'
os.chdir(proj_dir)

/tmp/ipykernel_1776/3030987161.py:14: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


Mounted at /gdrive


In [ ]:
#@title  Install the requisite packages and make sure you have a decent GPU (T4 suffices)
!pip install -q fair-esm biopython rdkit
!nvidia-smi

Sat May 23 21:43:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Observation- 1 : Non-VenomZone qualifier amiss

 Download the requisite raw data files and format the txt file.

According to [this](),
- Description of Additional supplementary files:

- **Supplementary data 1:** List of VEP predicted by APEX to have a median MIC ≤32 μmol<sup>-1</sup>.csv

- **Supplementary data 2:** List of VEP identified by APEX and filter criterion

The documentation for running inference states that: _"By running predict.py, species-specific antmicrobial activties (MICs) of peptides in test_seqs.txt will be generated and saved in Predicted_MICs.csv. To predict antmicrobial activties for novel peptides, you can replace the peptides in test_seqs.txt with the peptides of your interest. Alternatively, you can change line 84 of predict.py to the path of your peptide file. Please make sure that in this file, each line corresponds to a single peptide sequence (<= 50 amino acids in length)."_

In [ ]:
url_1='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM3_ESM.csv'

url_2='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM4_ESM.csv'

df_1=pd.read_csv(url_1)
df_2=pd.read_csv(url_2)

df_1.shape,df_2.shape

((4618, 13), (273, 16))

OK. So `df_1`has 4618 rows and df_2 has 273.

Thee only context in which 4618 and 273 appear is in the context of the "VenomZone(UniProtPK)" sub-dataset (Refer to the _Supplementary Table 1. Database-sourced venom protein and VEP candidates_ of the supplementary section found [here](https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM1_ESM.pdf)).

 Image of the table below:

 ![Supp Table 1](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/8d087f786b2151feb02947226f0285d75ad3789a/images/img_4618.png)


 https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM7_ESM.xlsx

# ERROR: I think there are only 55 unique peptide sequences amongst the 58 declared!!

In [33]:
url_58='https://github.com/vinayprabhu/APEX_reproduce/raw/refs/heads/main/df_final_58.tsv'
df_vep_58=pd.read_csv(url_58,sep='\t')
df_vep_58.Sequence.value_counts().head()

,count
Sequence,
KLLKIGLKSFARVLKKVL,2
KNKRFIRNLRSNLYQKIIKSTKSLL,2
KRRRASPLWKRRRFLSMLKARAK,2
LKLKSILGKLGVIL,1
RRVKRFKKFFMKLKKSVKKRVMKFFK,1


In [32]:
df_vep_58.Sequence.value_counts()[df_vep_58.Sequence.value_counts()>1].index

Index(['KLLKIGLKSFARVLKKVL', 'KNKRFIRNLRSNLYQKIIKSTKSLL',
       'KRRRASPLWKRRRFLSMLKARAK'],
      dtype='object', name='Sequence')

Yes indeed! Confirmed that ['KLLKIGLKSFARVLKKVL', 'KNKRFIRNLRSNLYQKIIKSTKSLL',
       'KRRRASPLWKRRRFLSMLKARAK'] are repeating via actual text search
       

![3 are duplicates](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/7d8e4ab1868a629ca07332fa8a7f5f41f37c54ce/images/img_58.png
)



In [ ]:
list_55=df_vep_58.Sequence.unique()
list_55

array(['KLLKIGLKSFARVLKKVL', 'WLGSALKIGAKLL', 'KLWNSKLARKIRTKGLKYVKNFAK',
       'LKLKSILGKLGVIL', 'RRVKRFKKFFMKLKKSVKKRVMKFFK', 'GKWLISSLVAKHL',
       'KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK', 'KLKKLRKWIYRIV',
       'FLKKIWRSKLVKRL', 'KRRRASPLWKRRRFLSMLKARAK', 'LTKWLGKLGVIL',
       'RKFKWGKLFSTAKKLYKKGKKLSKNKNFKKALK', 'KFLARLVFRKFILL',
       'KNKRFIRNLRSNLYQKIIKSTKSLL', 'KWLGKLGVILSHL',
       'RKFKWGKLFSTAKKLYKKGKKLSK', 'FIKKLWRSKLAKKLRAKGRELLK',
       'RRVKRFKKFFMKL', 'VNSFKIGGFIKKLWRSKLAKKLRAK', 'RFGSFLKKVWKSKLAKKL',
       'RRVKRFKKFFRKLKKSVKKRAKEFFK', 'RHRIVRTYIAKFGLK',
       'KRKGYLRLVPEERIWQKGLWWLRRLETDSDKLQK', 'LLHFSIWRSTVLRK',
       'RHRIVRTYIAKFGLKLNEFFQENENAWYFIRNIRKRVWEVKK', 'RRVKRFKKFFKKL',
       'QPRRVKRFKKFFKKLKNSVKKRAKKF', 'KRLRAKMLNSKFIKLIKR',
       'RRASPLWKRRRFLSMLKARAKRTGYK', 'PLWKRRRFLSMLKARAKR',
       'LRAKMLNSKFIKL', 'KLHGLLTRRSLKNFWKRNLYLR',
       'KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR', 'RLRAKMRNSKLFKLTKR',
       'RQEYPTKRLRAKMLNSKFIKLIKR', 'KKWRELSRLSRVLQIL'

# Error: The missing conoserver sequences!

![conoserver screenshot May 15 2026](https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/a73523b4807a1ccecc76bc346fa416239c7d00e6/images/conoserver.png)

The main issue:

- The final VEPs attributed to Conoserver were not found in the fa file!

In [48]:
#@title  First, let us carve out the 12 conoserver VEPs :

df_cs=df_vep_58.loc[df_vep_58.Peptide.str.contains('Conoserver', na=False), :]
concoserver_12=df_cs.Sequence.unique()
df_cs

,Peptide,Sequence
27,Conoserver-1,KRLRAKMLNSKFIKLIKR
28,Conoserver-2,KRRRASPLWKRRRFLSMLKARAK
29,Conoserver-3,RRASPLWKRRRFLSMLKARAKRTGYK
30,Conoserver-4,PLWKRRRFLSMLKARAKR
31,Conoserver-5,LRAKMLNSKFIKL
32,Conoserver-6,KLHGLLTRRSLKNFWKRNLYLR
33,Conoserver-7,KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR
34,Conoserver-10,RLRAKMRNSKLFKLTKR
35,Conoserver-12,RQEYPTKRLRAKMLNSKFIKLIKR
36,Conoserver-14,KKWRELSRLSRVLQIL


In [59]:
#@title Now, let us fetch all the conoserver data and extract the sequences!

def check_canonical_gt8(seq_c):
  """
  Function for filtering out the non-canonical sequences and sequences with length <8
  """
  CANONICAL = set("ACDEFGHIKLMNPQRSTVWY")
  iscanon=not (set(seq_c.upper()) - CANONICAL)
  isgt8=len(seq_c)>=8
  return iscanon & isgt8

from Bio import SeqIO
import requests
from io import StringIO

# URL to the raw FASTA file
url = "https://raw.githubusercontent.com/vinayprabhu/APEX_reproduce/main/conoserver_250206_protein.fa"

# Fetch the content and parse it using Biopython
response = requests.get(url)
fasta_data = StringIO(response.text)

# Define the expected columns
cols = [
    'conoserver_identifier', 'name', 'organism', 'protein type',
    'toxin class', 'gene_superfamily', 'cysteine_framework',
    'pharmacological_family', 'evidence'
]

# Extract header components and sequence
data = []
for record in SeqIO.parse(fasta_data, "fasta"):
    # Split the description line by '|'
    # Assuming the header follows the exact order of your column list
    header_parts = record.description.split('|')

    # Append the sequence to the list of header parts
    row = header_parts + [str(record.seq)]
    data.append(row)

# Create DataFrame
# Note: Added 'sequence' to columns since that is extracted from the FASTA
df_cs_raw = pd.DataFrame(data, columns=cols + ['sequence'])

# View the result
print(f'The current version has {df_cs_raw.shape[0]} seqs')

c_vec=df_cs_raw.sequence.apply(check_canonical_gt8)
# Apply the filter directly to the raw dataframe
df_cs_filt = df_cs_raw[c_vec].reset_index(drop=True)
# Verify the count
print(f"Filtered DataFrame shape: {df_cs_filt.shape}")

df_cs_filt.head(4)


The current version has 8523 seqs
Filtered DataFrame shape: (6603, 10)


,conoserver_identifier,name,organism,protein type,toxin class,gene_superfamily,cysteine_framework,pharmacological_family,evidence,sequence
0,P00005,PeIA precursor,Conus pergrandis,Precursor,conotoxin,A superfamily,,,,FDGRNAAANDKASDLVALTVRGCCSHPACSVNHPELCG
1,P00008,MII precursor,Conus magus,Precursor,conotoxin,A superfamily,,,,MGMRMMFTVFLLVVLATTVVSFPSDRASDGRNAAANDKASDVITLA...
2,P00009,SII precursor,Conus striatus,Precursor,conotoxin,A superfamily,,,,MGMRMMFTVFLLVVLATTVVSFPSDRASDGRDDEAKDERSDMHESD...
3,P00011,Ca1.1 precursor,Conus caracteristicus,Precursor,conotoxin,A superfamily,,,nucleic acid level,MGMRMMFTVFLLVVLATTVVSFTSDRASDGRNAAANAFDLIALIAR...


In [68]:
#@title Now, let us see how many of the final 58 sequences in df_cs are actually in df_cs_filt
##############################

def find_substring_locations(df, query_string):
  # Helper function for sub-string search
    # .map() replaces the deprecated .applymap()
    mask = df.map(lambda x: query_string in str(x))

    # Filter for rows containing the substring
    results = df[mask.any(axis=1)]

    return results
def print_list_in_red(my_list):
    # ANSI escape code for red text and reset
    RED = "\033[91m"
    RESET = "\033[0m"

    for element in my_list:
        print(f"{RED}{element}{RESET}")

def print_highlighted_substring(full_string, substring):
  # Helper function for sub-string printing with highlighting
    if not substring:
        print(full_string)
        return

    # ANSI escape code for green text and reset
    GREEN = "\033[92m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    # Find all starting indices of the substring
    start = 0
    indices = []
    while True:
        idx = full_string.find(substring, start)
        if idx == -1:
            break
        indices.append(idx)
        start = idx + 1

    if not indices:
        print(f"Substring '{substring}' not found.")
        return

    # Build the highlighted string
    highlighted = ""
    last_idx = 0
    for idx in indices:
        highlighted += full_string[last_idx:idx]
        # Wrap substring in ANSI codes
        highlighted += f"{GREEN}{BOLD}{substring}{RESET}"
        last_idx = idx + len(substring)

    highlighted += full_string[last_idx:]
    print(highlighted)
#################################

list_conoserver_notfound=[]
for seq_ in df_cs.Sequence.values:
  results=find_substring_locations(df_cs_filt,seq_)
  print(seq_)
  if(results.shape[0]>0):
    print(results[['conoserver_identifier', 'name', 'organism']])
    print_highlighted_substring(results['sequence'].values[0], seq_)
  else:
    print(f'{seq_} not found')
    list_conoserver_notfound.append(seq_)
  print('-------------------------')

print('Not found in list_conoserver: ')
print_list_in_red(list_conoserver_notfound)

KRLRAKMLNSKFIKLIKR
     conoserver_identifier              name               organism
5189                P07699  Ca6.13 precursor  Conus caracteristicus
MKLTCALIVAMLLLTACQLTTADASRGRQEYPTKRLRAKMLNSKFIKLIKRCAAPGASCSKYDNECCDACLLQYPNPPVC
-------------------------
KRRRASPLWKRRRFLSMLKARAK
     conoserver_identifier                  name          organism
5051                P06892  Con-ins G2 precursor  Conus geographus
MTTSSYFLLVALGLLLYVRQSFSTHEHTCQLDDPAHPQGKCGSDLVNYHEEKCEEEEARRGGTNDGGKKRRRASPLWKRRRFLSMLKARAKRTGYKGIACECCQHYCTDQEFINYCPPVTESSSSSSSAA
-------------------------
RRASPLWKRRRFLSMLKARAKRTGYK
     conoserver_identifier                  name          organism
5051                P06892  Con-ins G2 precursor  Conus geographus
MTTSSYFLLVALGLLLYVRQSFSTHEHTCQLDDPAHPQGKCGSDLVNYHEEKCEEEEARRGGTNDGGKKRRRASPLWKRRRFLSMLKARAKRTGYKGIACECCQHYCTDQEFINYCPPVTESSSSSSSAA
-------------------------
PLWKRRRFLSMLKARAKR
     conoserver_identifier                  name          organism
5051               

# Mismatch of MIC values via predict.py
Source: https://gitlab.com/machine-biology-group-public/apex/-/raw/main/predict.py


In [ ]:
%cd apex

/gdrive/MyDrive/venomics.ai/AI/VENOMICS_UPENN/apex


In [72]:
import os
import json
#from time import perf_counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math, copy, time
from torch.autograd import Variable
from scipy import stats
import pandas as pd
from sklearn.model_selection import KFold
import pickle
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import StepLR
import os.path
from Bio import SeqIO
import string
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from AMP_DL_model_twohead import AMP_model
#from propy.AAComposition import CalculateAADipeptideComposition
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy import stats
from utils import *
from scipy import sparse
import sys
from optparse import OptionParser
import copy
import pandas as pd


col = ['E. coli ATCC11775', 'P. aeruginosa PAO1', 'P. aeruginosa PA14', 'S. aureus ATCC12600', 'E. coli AIG221', 'E. coli AIG222', 'K. pneumoniae ATCC13883', 'A. baumannii ATCC19606', 'A. muciniphila ATCC BAA-835', 'B. fragilis ATCC25285', 'B. vulgatus ATCC8482', 'C. aerofaciens ATCC25986', 'C. scindens ATCC35704', 'B. thetaiotaomicron ATCC29148', 'B. thetaiotaomicron Complemmented', 'B. thetaiotaomicron Mutant', 'B. uniformis ATCC8492', 'B. eggerthi ATCC27754', 'C. spiroforme ATCC29900', 'P. distasonis ATCC8503', 'P. copri DSMZ18205', 'B. ovatus ATCC8483', 'E. rectale ATCC33656', 'C. symbiosum', 'R. obeum', 'R. torques', 'S. aureus (ATCC BAA-1556) - MRSA', 'vancomycin-resistant E. faecalis ATCC700802', 'vancomycin-resistant E. faecium ATCC700221', 'E. coli Nissle', 'Salmonella enterica ATCC 9150 (BEIRES NR-515)', 'Salmonella enterica (BEIRES NR-170)', 'Salmonella enterica ATCC 9150 (BEIRES NR-174)', 'L. monocytogenes ATCC 19111 (BEIRES NR-106)']

max_len = 52 # maximum peptide length

word2idx, idx2word = make_vocab()
emb, AAindex_dict = AAindex('aaindex1.csv', word2idx)
#Sourced from https://gitlab.com/machine-biology-group-public/apex/-/blob/main/aaindex1.csv?ref_type=heads
vocab_size = len(word2idx)
emb_size = np.shape(emb)[1]


model_num = 8
repeat_num = 5



f = open('best_key_list', 'r')
# Sourced from https://gitlab.com/machine-biology-group-public/apex/-/blob/main/best_key_list?ref_type=heads
lines = f.readlines()
f.close()

model_list = []
for line in lines:
  parsed = line.strip('\n').strip('\r')
  model_list.append(parsed)


all_list = []
ensemble_num = model_num * repeat_num

deep_model_list = []
for a_model_name in model_list:
  for a_en in range(repeat_num):
    key = 'trained_all_model_'+a_model_name+'_ensemble_'+str(a_en)

    #model = torch.load('./trained_models/'+key)
    model = torch.load('./trained_models/'+key, weights_only=False)
    model.eval()
    deep_model_list.append(model)


In [77]:
seq_list=df_vep_58.Sequence.unique()
seq_list

array(['KLLKIGLKSFARVLKKVL', 'WLGSALKIGAKLL', 'KLWNSKLARKIRTKGLKYVKNFAK',
       'LKLKSILGKLGVIL', 'RRVKRFKKFFMKLKKSVKKRVMKFFK', 'GKWLISSLVAKHL',
       'KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK', 'KLKKLRKWIYRIV',
       'FLKKIWRSKLVKRL', 'KRRRASPLWKRRRFLSMLKARAK', 'LTKWLGKLGVIL',
       'RKFKWGKLFSTAKKLYKKGKKLSKNKNFKKALK', 'KFLARLVFRKFILL',
       'KNKRFIRNLRSNLYQKIIKSTKSLL', 'KWLGKLGVILSHL',
       'RKFKWGKLFSTAKKLYKKGKKLSK', 'FIKKLWRSKLAKKLRAKGRELLK',
       'RRVKRFKKFFMKL', 'VNSFKIGGFIKKLWRSKLAKKLRAK', 'RFGSFLKKVWKSKLAKKL',
       'RRVKRFKKFFRKLKKSVKKRAKEFFK', 'RHRIVRTYIAKFGLK',
       'KRKGYLRLVPEERIWQKGLWWLRRLETDSDKLQK', 'LLHFSIWRSTVLRK',
       'RHRIVRTYIAKFGLKLNEFFQENENAWYFIRNIRKRVWEVKK', 'RRVKRFKKFFKKL',
       'QPRRVKRFKKFFKKLKNSVKKRAKKF', 'KRLRAKMLNSKFIKLIKR',
       'RRASPLWKRRRFLSMLKARAKRTGYK', 'PLWKRRRFLSMLKARAKR',
       'LRAKMLNSKFIKL', 'KLHGLLTRRSLKNFWKRNLYLR',
       'KRGRASPLWQRRGFLSKLKARAKRNGAFHLPR', 'RLRAKMRNSKLFKLTKR',
       'RQEYPTKRLRAKMLNSKFIKLIKR', 'KKWRELSRLSRVLQIL'

In [91]:
import time
t=time.time()


ensemble_counter = 0
for ensemble_id in range(ensemble_num):

	AMP_model = deep_model_list[ensemble_id].cuda().eval()

	data_len = len(seq_list)
	batch_size = 3000 #change according to your GPU memory
	for i in range(int(math.ceil(data_len/float(batch_size)))):
		# if (i*batch_size) % 1000 == 0:
		# 	print ('progress', i*batch_size, data_len)

		seq_batch = seq_list[i*batch_size:(i+1)*batch_size]
		seq_rep, _, _ = onehot_encoding(seq_batch, max_len, word2idx)

		X_seq = torch.LongTensor(seq_rep).cuda()


		AMP_pred_batch = AMP_model(X_seq).cpu().detach().numpy()
		AMP_pred_batch = 10**(6-AMP_pred_batch) #transform back to MICs

		if i == 0:
			AMP_pred = AMP_pred_batch
		else:
			AMP_pred = np.vstack([AMP_pred, AMP_pred_batch])

	if ensemble_id == 0:
		AMP_sum = AMP_pred
	else:
		AMP_sum += AMP_pred
	ensemble_counter += 1

AMP_pred = AMP_sum / float(ensemble_counter)
print('time taken', time.time()-t)
df_vep_mic = pd.DataFrame(data=AMP_pred, columns=col, index=seq_list)
df_vep_mic.median(axis=1)

time taken 0.398787260055542


,0
KLLKIGLKSFARVLKKVL,18.493744
WLGSALKIGAKLL,17.443626
KLWNSKLARKIRTKGLKYVKNFAK,46.957985
LKLKSILGKLGVIL,12.831366
RRVKRFKKFFMKLKKSVKKRVMKFFK,56.134571
GKWLISSLVAKHL,15.641453
KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK,66.547585
KLKKLRKWIYRIV,37.603188
FLKKIWRSKLVKRL,27.004911
KRRRASPLWKRRRFLSMLKARAK,56.474556


In [84]:
df_mic_56 = df_vep_mic.reset_index().rename(columns={'index': 'Sequence'})
df_mic_56

,Sequence,E. coli ATCC11775,P. aeruginosa PAO1,P. aeruginosa PA14,S. aureus ATCC12600,E. coli AIG221,E. coli AIG222,K. pneumoniae ATCC13883,A. baumannii ATCC19606,A. muciniphila ATCC BAA-835,...,R. obeum,R. torques,S. aureus (ATCC BAA-1556) - MRSA,vancomycin-resistant E. faecalis ATCC700802,vancomycin-resistant E. faecium ATCC700221,E. coli Nissle,Salmonella enterica ATCC 9150 (BEIRES NR-515),Salmonella enterica (BEIRES NR-170),Salmonella enterica ATCC 9150 (BEIRES NR-174),L. monocytogenes ATCC 19111 (BEIRES NR-106)
0,KLLKIGLKSFARVLKKVL,7.630094,12.327852,17.703650,23.381571,10.479136,9.201456,22.415638,2.904467,6.383162,...,346.442291,143.101898,24.319881,40.748459,5.034935,514.965027,8.305558,235.542328,261.745758,15.221005
1,WLGSALKIGAKLL,12.725700,44.431915,56.736542,11.008323,11.277797,8.040901,21.578999,6.238828,30.969669,...,194.455643,47.988377,13.541011,40.069790,5.775425,354.395935,15.283323,323.942871,405.748230,20.224804
2,KLWNSKLARKIRTKGLKYVKNFAK,14.614248,9.662188,12.731038,41.280724,8.959637,11.542986,48.083843,7.042480,7.175234,...,462.761627,321.855591,53.418976,95.589005,7.445621,336.862549,4.462077,222.412766,156.180191,10.158847
3,LKLKSILGKLGVIL,9.679654,28.665686,36.115726,14.463969,10.064558,6.648721,18.667576,6.695969,8.740446,...,118.953995,40.000294,13.272173,32.564507,5.365795,245.696167,16.972363,128.598923,126.641281,24.514133
4,RRVKRFKKFFMKLKKSVKKRVMKFFK,15.252353,8.151583,13.043551,53.608166,13.172803,13.664485,92.007004,12.239097,3.642266,...,578.644897,373.900055,57.094898,95.394394,8.477136,447.937592,4.666789,153.183640,188.099930,12.393736
5,GKWLISSLVAKHL,9.333745,35.509205,44.693993,13.826393,8.518499,6.608459,16.788589,3.842312,18.526373,...,214.066330,52.649384,17.562584,47.315865,6.320087,162.949921,11.009680,124.680275,114.633347,14.756948
6,KRLKGFAKKLWNSKLARKIRTKGLKYVKNFAK,20.012367,11.588827,14.706251,44.356834,11.053272,14.438757,66.986977,11.407370,8.143622,...,499.354553,397.320770,61.039051,97.165405,7.910892,406.996185,4.791270,171.932999,167.583969,10.755083
7,KLKKLRKWIYRIV,15.005010,14.140190,23.003414,52.312584,13.605214,14.748598,69.368225,12.723369,2.595351,...,407.990234,280.034485,46.796757,90.778450,11.951981,249.537918,11.988327,77.700813,90.218796,25.637127
8,FLKKIWRSKLVKRL,10.645987,12.230603,20.745794,26.754894,15.425552,11.063894,59.602131,6.821894,5.297017,...,462.897705,222.014084,28.743210,57.977730,7.871366,250.341873,12.121559,170.172424,170.256927,24.756218
9,KRRRASPLWKRRRFLSMLKARAK,15.617238,13.151022,15.691370,51.631752,13.443766,12.150095,63.456982,8.165344,11.383955,...,540.185730,310.739410,69.627792,92.638756,8.241979,399.135193,5.169033,263.933807,201.098190,11.031645


In [105]:
(df_mic_56.iloc[:,1:].median(axis=1).values>32).sum(),(df_mic_56.iloc[:,1:].median(axis=1).values>32).mean()

(np.int64(42), np.float64(0.7636363636363637))

In [85]:
df_mic_56.to_csv('df_mic_56.csv', index=False)

In [86]:
# Supplementary data 1:List of VEP predicted by APEX to have a median MIC ≤32 μmol<sup>-1</sup>.csv
url_1='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM3_ESM.csv'

# Supplementary data 2:List of VEP identified by APEX and filter criterion
url_2='https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-60051-6/MediaObjects/41467_2025_60051_MOESM4_ESM.csv'

df_1=pd.read_csv(url_1)
df_2=pd.read_csv(url_2)

df_1.shape,df_2.shape

((4618, 13), (273, 16))

In [96]:
seq_vep_4618=df_1.Sequence.values
seq_vep_4618

array(['VNWKKVLGKIIKVAK', 'VNWKKILGKIIKVAK', 'VNWKKILGKIIKVA', ...,
       'SFKFGSFIKRMWRSKLAKKLRAK', 'KGFAKKLWNSKLARKIRTKG',
       'KFGGFLKKMWKSKLAKKLRAKGKQMLKEYANKVL'], dtype=object)

In [97]:
t=time.time()


ensemble_counter = 0
for ensemble_id in range(ensemble_num):

	AMP_model = deep_model_list[ensemble_id].cuda().eval()

	data_len = len(seq_vep_4618)
	batch_size = 3000 #change according to your GPU memory
	for i in range(int(math.ceil(data_len/float(batch_size)))):
		# if (i*batch_size) % 1000 == 0:
		# 	print ('progress', i*batch_size, data_len)

		seq_batch = seq_vep_4618[i*batch_size:(i+1)*batch_size]
		seq_rep, _, _ = onehot_encoding(seq_batch, max_len, word2idx)

		X_seq = torch.LongTensor(seq_rep).cuda()


		AMP_pred_vep_4618_batch = AMP_model(X_seq).cpu().detach().numpy()
		AMP_pred_vep_4618_batch = 10**(6-AMP_pred_vep_4618_batch) #transform back to MICs

		if i == 0:
			AMP_pred_vep_4618 = AMP_pred_vep_4618_batch
		else:
			AMP_pred_vep_4618 = np.vstack([AMP_pred_vep_4618, AMP_pred_vep_4618_batch])

	if ensemble_id == 0:
		AMP_sum_vep_4618 = AMP_pred_vep_4618
	else:
		AMP_sum_vep_4618 += AMP_pred_vep_4618
	ensemble_counter += 1

AMP_pred_vep_4618 = AMP_sum_vep_4618 / float(ensemble_counter)
print('time taken', time.time()-t)
df_vep_mic_4618 = pd.DataFrame(data=AMP_pred_vep_4618, columns=col, index=seq_vep_4618)
df_vep_mic_4618.median(axis=1)

time taken 13.950983047485352


,0
VNWKKVLGKIIKVAK,9.386429
VNWKKILGKIIKVAK,8.417408
VNWKKILGKIIKVA,9.908989
VNWKKILGKIIKVVK,8.617786
VNWKKVLGKIIKVA,12.370358
...,...
LKELWTKIKGAGKAVLGKIKGLL,32.866238
KRLWRNWEDLELRQLLNEFAENQREKRLWRNWERRQ,65.837418
SFKFGSFIKRMWRSKLAKKLRAK,57.520020
KGFAKKLWNSKLARKIRTKG,57.115543


In [101]:
(df_vep_mic_4618.median(axis=1).values>32).sum(),(df_vep_mic_4618.median(axis=1).values>32).mean()

(np.int64(4266), np.float64(0.9237765266349068))

In [102]:
df_vep_mic_4618.to_csv('df_mic_4618.csv', index=False)